# The Chain Rule
**Topic:** Calculus for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Explain** what a composed function is and recognize when two functions are chained together
- **Apply** the chain rule to compute the derivative of a composed function
- **Interpret** how backpropagation in a neural network is the chain rule applied layer by layer

> **Tip:** Start by moving the **x slider** and watch both charts update simultaneously: the left chart shows the inner function g(x), and the right chart shows the outer function f(g(x)). Notice how a change in x ripples through both functions before producing the final output.

---
## How we got here

In *03: Derivatives* we learned to differentiate simple functions. But real-world functions are rarely simple. The sigmoid activation function is e raised to a power, divided by a sum. The loss function of a neural network involves dozens of nested operations. The chain rule is the single formula that lets us differentiate any composed function, no matter how deeply nested.

---
## Why this matters for data science

Backpropagation, the algorithm that trains every neural network, is the chain rule. When a network makes a prediction, it passes data through many layers, each applying a function. When we compute the gradient of the loss with respect to the first layer's weights, we must differentiate through every subsequent layer using the chain rule. Without the chain rule there is no backpropagation, without backpropagation there is no gradient, and without the gradient there is no learning.

---
## Try it yourself

In [2]:
out = Output()

def _sig(u):
    return 1.0 / (1.0 + np.exp(-np.asarray(u, dtype=float)))

_FUNCS = {
    "sin(x²)": {
        "g": lambda x: x**2, "dg": lambda x: 2.0*x,
        "f": lambda u: np.sin(u), "df": lambda u: np.cos(u),
        "g_label": "inner: g(x) = x²", "f_label": "outer: f(u) = sin(u)",
    },
    "e^(2x)": {
        "g": lambda x: 2.0*x, "dg": lambda x: 2.0 + 0.0*x,
        "f": lambda u: np.exp(u), "df": lambda u: np.exp(u),
        "g_label": "inner: g(x) = 2x", "f_label": "outer: f(u) = eᵘ",
    },
    "ln(x²+1)": {
        "g": lambda x: x**2 + 1, "dg": lambda x: 2.0*x,
        "f": lambda u: np.log(u), "df": lambda u: 1.0/u,
        "g_label": "inner: g(x) = x²+1", "f_label": "outer: f(u) = ln(u)",
    },
    "sigmoid(3x−1)": {
        "g": lambda x: 3.0*x - 1.0, "dg": lambda x: 3.0 + 0.0*x,
        "f": lambda u: _sig(u), "df": lambda u: _sig(u)*(1-_sig(u)),
        "g_label": "inner: g(x) = 3x−1", "f_label": "outer: f(u) = σ(u)",
    },
}

_x_pts = np.linspace(-2.5, 2.5, 300)

func_dd = Dropdown(options=list(_FUNCS.keys()), value="sin(x²)",
    description="Function:", style={"description_width": "80px"},
    layout=widgets.Layout(width="340px"))

x_sl = FloatSlider(value=1.0, min=-2.5, max=2.5, step=0.05,
    description="x =", style={"description_width": "40px"},
    layout=widgets.Layout(width="400px"))

def _tangent(xc, yc, slope, halfspan, x_lo, x_hi):
    x0, x1 = max(xc - halfspan, x_lo), min(xc + halfspan, x_hi)
    return [x0, x1], [yc + slope*(x0 - xc), yc + slope*(x1 - xc)]

def render(change=None):
    fn = _FUNCS[func_dd.value]
    x0 = float(x_sl.value)

    g0  = float(fn["g"](x0))
    h0  = float(fn["f"](g0))
    dg0 = float(fn["dg"](x0))
    df0 = float(fn["df"](g0))
    dh0 = dg0 * df0

    g_curve = fn["g"](_x_pts)
    u_lo, u_hi = float(np.min(g_curve)), float(np.max(g_curve))
    if "ln(u)" in fn["f_label"]:
        u_lo = max(u_lo, 1e-3)
    u_pts = np.linspace(u_lo, u_hi, 300)
    f_curve = fn["f"](u_pts)

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14,
        subplot_titles=[fn["g_label"], fn["f_label"]])

    # ----- LEFT: inner g(x) + tangent (slope g') -----
    fig.add_trace(go.Scatter(x=_x_pts, y=g_curve, mode="lines",
        line=dict(color=PALETTE["primary"], width=2.5), showlegend=False), 1, 1)
    tx, ty = _tangent(x0, g0, dg0, 0.9, _x_pts.min(), _x_pts.max())
    fig.add_trace(go.Scatter(x=tx, y=ty, mode="lines",
        line=dict(color=PALETTE["secondary"], width=3), showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=[x0], y=[g0], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=12,
                    line=dict(width=2, color="white")), showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=[x0, x0], y=[0, g0], mode="lines",
        line=dict(color=PALETTE["accent"], width=1.5, dash="dot"),
        showlegend=False), 1, 1)
    # inline slope label, floated just past the tangent's upper end
    fig.add_annotation(x=tx[1], y=ty[1], xref="x1", yref="y1",
        text=f"slope g'(x) = {dg0:.2f}", showarrow=False,
        font=dict(color=PALETTE["secondary"], size=12, family=FONT["family"]),
        xanchor="left", xshift=4, bgcolor="rgba(255,255,255,0.7)")

    # ----- RIGHT: outer f(u) on its own axis + tangent (slope f') -----
    fig.add_trace(go.Scatter(x=u_pts, y=f_curve, mode="lines",
        line=dict(color=PALETTE["primary"], width=2.5), showlegend=False), 1, 2)
    span = (u_hi - u_lo) * 0.18
    tx2, ty2 = _tangent(g0, h0, df0, span, u_lo, u_hi)
    fig.add_trace(go.Scatter(x=tx2, y=ty2, mode="lines",
        line=dict(color=PALETTE["secondary"], width=3), showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=[g0], y=[h0], mode="markers",
        marker=dict(color=PALETTE["secondary"], size=12,
                    line=dict(width=2, color="white")), showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=[g0, g0], y=[float(np.min(f_curve)), h0], mode="lines",
        line=dict(color=PALETTE["accent"], width=1.5, dash="dot"),
        showlegend=False), 1, 2)
    fig.add_annotation(x=tx2[1], y=ty2[1], xref="x2", yref="y2",
        text=f"slope f'(u) = {df0:.2f}", showarrow=False,
        font=dict(color=PALETTE["secondary"], size=12, family=FONT["family"]),
        xanchor="left", xshift=4, bgcolor="rgba(255,255,255,0.7)")
    # label the relay handoff on the outer x-axis
    fig.add_annotation(x=g0, y=float(np.min(f_curve)), xref="x2", yref="y2",
        text=f"u = g(x) = {g0:.2f}", showarrow=True, arrowhead=2, ax=0, ay=24,
        font=dict(color=PALETTE["accent"], size=11, family=FONT["family"]))

    layout = base_layout(
        title=f"x = {x0:.2f}  →  g(x) = {g0:.3f}  →  f(g(x)) = {h0:.3f}",
        xaxis_title="", yaxis_title="")
    layout.update(height=380, showlegend=False)
    fig.update_layout(layout)
    fig.update_xaxes(title_text="x  (input)", row=1, col=1)
    fig.update_xaxes(title_text="u = g(x)  (relayed input)", row=1, col=2)

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))
        display(HTML(
            f'<div style="font-family:{FONT["family"]}; font-size:14px; '
            f'padding:10px 18px; background:#f8f9fa; border-radius:6px; margin-top:4px;">'
            f'The slider drops a point onto the <b>inner</b> curve; its output '
            f'<b>u = {g0:.3f}</b> is relayed across (dotted lines) to become the '
            f'<b>outer</b> curve\'s input. The chain rule multiplies the two tangent slopes:'
            f'<br><br>'
            f'<span style="color:{PALETTE["secondary"]}"><b>inner slope g\'(x) = {dg0:.3f}</b></span>'
            f'&ensp;&times;&ensp;'
            f'<span style="color:{PALETTE["secondary"]}"><b>outer slope f\'(u) = {df0:.3f}</b></span>'
            f'&ensp;=&ensp;'
            f'<b style="font-size:15px;">dh/dx = {dh0:.3f}</b>'
            f'</div>'))

func_dd.observe(render, names="value")
x_sl.observe(render, names="value")
display(VBox([HBox([func_dd, x_sl]), out]))
render()

---
## What's happening?

A composed function feeds the output of one function into another. If g(x) = x² and f(u) = sin(u), then f(g(x)) = sin(x²). The widget above shows this as a **relay in three steps**:

1. The slider sets an input **x**, which lands on the **inner** curve g — the point's height is g(x).
2. That height is **relayed across** (the dotted lines) to become the **input** of the outer curve f. This is the key move: the inner function's *output* is the outer function's *input*, which is why the right panel's horizontal axis is labeled **u = g(x)**, not x.
3. The point lands on the outer curve at height f(g(x)) — the final output.

The chain rule asks: if I nudge x a little, how much does the final output move? The nudge has to travel the same relay. First it gets **scaled by the inner slope** g'(x) — the tangent on the left panel. Then that already-scaled nudge gets **scaled again by the outer slope** f'(u) at the relayed point — the tangent on the right panel. Two scalings in sequence means the rates **multiply**:

$$\frac{dy}{dx} = \underbrace{\frac{df}{du}}_{\text{outer slope at } u=g(x)} \cdot \underbrace{\frac{dg}{dx}}_{\text{inner slope at } x}$$

Watch the two tangent lines as you drag. When the inner tangent is steep and the outer is shallow (or vice versa), the final slope is the product — a steep stage and a flat stage partly cancel; two steep stages compound. That multiplication of slopes is the entire chain rule, and it's why a single near-flat stage anywhere in the chain can flatten the whole thing.

| Composed function | Inner g(x) | Outer f(u) | Chain rule result |
|------------------|-----------|-----------|------------------|
| sin(x²) | g(x) = x², g'(x) = 2x | f(u) = sin(u), f'(u) = cos(u) | cos(x²) · 2x |
| e^(2x) | g(x) = 2x, g'(x) = 2 | f(u) = eᵘ, f'(u) = eᵘ | e^(2x) · 2 |
| sigmoid(3x−1) | g(x) = 3x−1, g'(x) = 3 | f(u) = 1/(1+e⁻ᵘ) | σ(3x−1)·(1−σ(3x−1))·3 |
| ln(x² + 1) | g(x) = x²+1, g'(x) = 2x | f(u) = ln(u), f'(u) = 1/u | 2x/(x²+1) |

### Why backpropagation is the chain rule

A neural network stacks these relays many deep: each layer takes the previous layer's output as its input, applies a weight and an activation, and passes its output forward. To find how much the first layer's weight affected the loss, the nudge has to travel back through every relay — and at each one it's scaled by that stage's local slope. Backpropagation is just this relay run in reverse, multiplying one local slope per layer. The interactive cell further down lets you watch what happens when those slopes are small: the product collapses, and the gradient vanishes.

---
## Real-world example: Backpropagation through a two-layer network

The chart traces the chain rule computation through a minimal neural network with one hidden layer. Each arrow in the forward pass becomes a factor in the chain rule product during the backward pass.

Notice:

- **Notice:** The forward pass (left to right) computes the prediction; the backward pass (right to left) computes gradients by applying the chain rule at each step in reverse
- **Notice:** The gradient at each layer is the product of all the gradients to its right; if any one of those factors is very small (for example, a saturated sigmoid), the product can vanish toward zero, this is the vanishing gradient problem
- **Notice:** Deeper networks have more chain rule factors to multiply; this is why vanishing gradients are more severe in deep networks, and why ReLU (whose derivative is 1 for positive inputs) alleviates the problem

> **Discussion question:** If every layer in a deep network used a sigmoid activation with a saturated output (derivative ≈ 0), what would happen to the gradient by the time it reaches the first layer? How does this explain why training very deep networks with sigmoid activations was so difficult before ReLU was widely adopted?

In [3]:
# ── Vanishing gradients, drawn as a network: watch the backward pass fade ─────
out_net = Output()

depth_sl = IntSlider(value=4, min=2, max=6, step=1,
    description="Hidden layers:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="430px"))

sat_sl = FloatSlider(value=2.5, min=0.0, max=4.0, step=0.25,
    description="Saturation (bias):",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="430px"))

net_caption = widgets.HTML(layout=widgets.Layout(width="760px"))

X_VAL, LAYER_W, NODES = 1.0, 0.6, 3   # nodes drawn per layer (cosmetic; math uses 1 rep unit)

def _hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def render_net(change=None):
    n_layers = int(depth_sl.value)
    bias = float(sat_sl.value)

    # Forward pass (one representative unit per layer)
    a = X_VAL
    acts = []
    for _ in range(n_layers):
        a = 1 / (1 + np.exp(-(LAYER_W * a + bias)))
        acts.append(a)
    yhat = acts[-1]

    # Backward pass: gradient magnitude arriving at each layer boundary (L→R)
    g = 2 * abs(yhat - 0.0)
    mags = [g]
    sig_derivs = []
    for i in reversed(range(n_layers)):
        sd = acts[i] * (1 - acts[i]); sig_derivs.append(sd)
        g *= sd
        if i > 0: g *= LAYER_W
        mags.append(g)
    mags = mags[::-1]                 # input-side → output-side
    sig_derivs = sig_derivs[::-1]
    gmax = max(mags) if max(mags) > 0 else 1.0
    opacity = [max(0.04, m / gmax) for m in mags]   # arrow strength per layer gap

    # Layer x-positions: input, hidden×n, output
    n_cols = n_layers + 2
    xs = list(range(n_cols))
    ys = np.linspace(-1, 1, NODES)

    fig = go.Figure()
    base_rgb = _hex_to_rgb(PALETTE["primary"])
    red_rgb  = _hex_to_rgb(PALETTE["secondary"])

    # ---- Backward gradient arrows between layers (right→left), fading as they go ----
    # opacity[k] is the gradient strength in the gap to the LEFT of column k+1
    for col in range(n_cols - 1):
        op = opacity[col]                       # strength entering this gap from the right
        width = 1 + 6 * op
        for ya in ys:
            for yb in ys:
                fig.add_trace(go.Scatter(
                    x=[xs[col+1], xs[col]], y=[yb, ya], mode="lines",
                    line=dict(color=f"rgba({red_rgb[0]},{red_rgb[1]},{red_rgb[2]},{op:.3f})",
                              width=width),
                    hoverinfo="skip", showlegend=False))

    # ---- Nodes ----
    for col in range(n_cols):
        is_io = col == 0 or col == n_cols - 1
        label = ("x" if col == 0 else "ŷ" if col == n_cols-1 else f"σ")
        fig.add_trace(go.Scatter(
            x=[xs[col]]*NODES, y=ys, mode="markers",
            marker=dict(size=34,
                        color=PALETTE["surface"] if is_io else PALETTE["primary"],
                        line=dict(color=PALETTE["primary"], width=2)),
            hoverinfo="skip", showlegend=False))
        # layer caption under each hidden layer: its sigmoid derivative
        if 1 <= col <= n_layers:
            sd = sig_derivs[col-1]
            fig.add_annotation(x=xs[col], y=-1.45, text=f"σ' = {sd:.3f}",
                showarrow=False, font=dict(size=11, family=FONT["family"],
                color=PALETTE["secondary"] if sd < 0.1 else PALETTE["muted"]))

    # Endpoint labels
    fig.add_annotation(x=0, y=1.5, text="input", showarrow=False,
        font=dict(size=12, family=FONT["family"], color=PALETTE["muted"]))
    fig.add_annotation(x=n_cols-1, y=1.5, text="output / loss", showarrow=False,
        font=dict(size=12, family=FONT["family"], color=PALETTE["muted"]))
    fig.add_annotation(x=0.5, y=-1.9,
        text="← backward pass: arrow fades as the gradient vanishes",
        showarrow=False, xanchor="left",
        font=dict(size=12, family=FONT["family"], color=PALETTE["secondary"]))

    layout = base_layout(
        title=f"{n_layers} sigmoid layers, saturation={bias:.2f}  |  "
              f"gradient reaching layer 1 = {mags[0]:.2e}",
        xaxis_title="", yaxis_title="")
    layout.update(height=420, showlegend=False,
        xaxis=dict(visible=False, range=[-0.6, n_cols-0.4]),
        yaxis=dict(visible=False, range=[-2.1, 1.8]))
    fig.update_layout(layout)

    # Caption
    reaches = mags[0]
    verdict = ("essentially zero — the first layer barely learns"
               if reaches < 1e-2 else
               "still sizable — this network would train fine")
    net_caption.value = (
        f"<div style='font-size:13px; line-height:1.5; padding:6px 2px'>"
        f"Each red arrow is the gradient flowing <b>backward</b>. Its strength is the "
        f"running chain-rule product, and it <b>fades left</b> because every sigmoid layer "
        f"multiplies in a derivative capped at 0.25 (peak, at the curve's center) — here "
        f"σ' ≈ {min(sig_derivs):.3f} at the most saturated layer. By the first layer the "
        f"gradient is <b>{reaches:.2e}</b> — {verdict}. "
        f"Raise <b>saturation</b> (push the sigmoids toward their flat tails) or add "
        f"<b>layers</b> and watch the arrows wash out faster. ReLU fixes this: its "
        f"derivative is 1, not ≤0.25, so the product doesn't decay.</div>")

    with out_net:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

depth_sl.observe(render_net, names="value")
sat_sl.observe(render_net, names="value")
display(VBox([HBox([depth_sl, sat_sl]), out_net, net_caption]))
render_net()

### Chain rule in common ML operations

| Composed function | Context | Chain rule factors |
|------------------|---------|-------------------|
| Loss(sigmoid(Xw)) | Logistic regression | dLoss/d(sigmoid) · sigmoid' · X |
| Loss(f_L(f_{L-1}(...f_1(x)))) | Neural network | Product of L gradient terms |
| MSE(softmax(Xw)) | Multiclass classifier | dMSE/d(softmax) · d(softmax)/d(Xw) · X |
| L2(weights) + MSE(predictions) | Ridge regression loss | Derivative splits by sum rule, chain rule on each term |

---
## Key takeaway

> **The chain rule says: multiply the derivatives of each nested function in sequence; backpropagation is the chain rule applied in reverse through every layer of a neural network to compute how much each weight contributed to the error.**

---
*Next up: Partial Derivatives — how to differentiate when a function depends on more than one variable, which is every real model with more than one weight*